<a href="https://colab.research.google.com/github/agarwal-ritika/coding-samples-/blob/main/Grant_monitoring_tool_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#instructions
# 1) check that relevant data in gpex has been updated
# particularly grants entered, status and dates updated including effectiveness, and GOO/CPOs have been updated
# 2) prepare files for upload
#download the relevant files from GPEX
#create the utilization last updated file
#check small grants in megatable such that none are pending, and if any are pending update both in file and in gpex
#delete duplicate grant ID rows in RM dashboard and megatable
#in bulk export/gpex data dump delete blank first column and first row
#megatable you can leave as is with first blank row there
# 3) directly in the code below update as_of_date_input and file names
# 4) run script
# 5) download the excel output that pops up
# 6) adjust and format output file
#if certain grants don't have a date or have N/A for Date Utilization Last Updated (or if grant start date is after date last utilized) you have to manually put N/A for the below columns:
# Date Utilization Last Updated; Cumulative Utilized; Percentage utilized cumulatively; Percentage utilized in last 12 months; Utilization lag; Utilization Status
#drop actual closing date columns
#format dates and numbers
#remove Grant Status from pending tab
#clean headers with asterisks
#remember to delete or cleanup the notes in the pending tab
#double check the grants in proposal submitted status - if there are some might have to manually add

#warning
#script was created with certain column names outputted from GPEX
#some gpex column names are changing so you might get error if a column name in gpex changes
#if that's the case scroll all the way to the bottom where you'll find error details including which columns it can't find
#for example if it can't find the GOO column then you'll have to look through the files you uploaded and see what the new column name is
#for example if it's CPO now then you'd have to carefully change GOO references in the script to CPO

import pandas as pd
from datetime import datetime
from pandas.tseries.offsets import MonthBegin

# setup
as_of_date_input   = '2026-01-31'
as_of_date         = pd.to_datetime(as_of_date_input)

input_file         = '/content/Implementation Grants.xlsx'    # implementation datasheet
new_util_file      = '/content/GPE 3_2_2026.xlsx'    # gpex data dump
date_upload_file   = '/content/Date last utilized.xlsx'    # manually created date utilization last reported file with only Grant ID + Date Utilization Last Updated
rm_dashboard_file  = '/content/RM Dashboard.xlsx'    # RM dashboard
mega_file          = '/content/Mega_Data_3_2_2026.xlsx'    # mega table


# part 1: implementation grants sheet
df = pd.read_excel(input_file)

keep_cols = [
    "Cluster",  # → region
    "Country Name", "Grant ID", "Grant Type", "P#",
    "Grant Agent", "Grant Operations Officer", "Operating Model",
    "Grant Approval Date", "*Actual Grant Start Date", "*Actual Grant Closing Date",
    "*Expected Grant Closing Date", "Grant Amount",
    "Cumulative Utilized",  # now sourced from input_file
    "Last Progress Report Submitted/ Received Date", "Link to Progress Report",
    "Progress Report Submission Status",
    "If Extended How many Times", "Cumulative extension (in months)"
]
df = df[keep_cols].copy()
df.rename(columns={'Cluster': 'Region'}, inplace=True)
df['Operating Model'] = df['Operating Model'].fillna(2020)

# parse dates
for c in [
    "Grant Approval Date", "*Actual Grant Start Date",
    "*Actual Grant Closing Date", "*Expected Grant Closing Date"
]:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# filter active
mask_active = (
    (df["*Actual Grant Start Date"] <= as_of_date) &
    (
        (df["*Actual Grant Closing Date"].isna() & (df["*Expected Grant Closing Date"] > as_of_date)) |
        (df["*Actual Grant Closing Date"] > as_of_date)
    )
)
df = df.loc[mask_active].reset_index(drop=True)
df['Cumulative extension (in months)'] = df['Cumulative extension (in months)'].round(1)

# drop any grants whose type includes "(AF)"
df = df.loc[~df['Grant Type'].str.contains(r'\(AF\)', na=False)].reset_index(drop=True)

# merge in date utilization last updated
df_dates = pd.read_excel(date_upload_file)
df_dates.columns = df_dates.columns.str.strip()
df_dates['Date of last utilization'] = pd.to_datetime(
    df_dates['Date of last utilization'], errors='coerce'
)
df = df.merge(
    df_dates[['Grant ID', 'Date of last utilization']].rename(columns={'Date of last utilization': 'Date Utilization Last Updated'}),
    on='Grant ID', how='left'
)

# last 12 month share calculation using new_util_file
df_new = pd.read_excel(new_util_file)
df_new.rename(columns={
    df_new.columns[3]: "Grant ID",
    df_new.columns[79]: "Util Date",
    df_new.columns[80]: "Util Amount"
}, inplace=True)
df_new["Util Date"] = pd.to_datetime(df_new["Util Date"], errors='coerce')

date_lookup = df.set_index('Grant ID')['Date Utilization Last Updated'].to_dict()

def compute_last12(group):
    gid = group.name
    latest = date_lookup.get(gid, group['Util Date'].max())
    start  = latest - MonthBegin(12)
    return group.loc[
        (group['Util Date'] >= start) & (group['Util Date'] <= latest),
        'Util Amount'
    ].sum()

series12  = df_new.groupby('Grant ID').apply(compute_last12, include_groups=False)
df_last12 = series12.reset_index(name='Amount utilized in last 12 months')
df = df.merge(df_last12, on='Grant ID', how='left')

# percentage utilized in last 12 months (as fraction of grant)
df['Percentage utilized in last 12 months'] = (
    df['Amount utilized in last 12 months'] / df['Grant Amount']
)
df.drop(columns=['Amount utilized in last 12 months'], inplace=True)

# output N/A if under 12 months since start to last update
mask_under12 = (
    (df['Date Utilization Last Updated'] - df['*Actual Grant Start Date'])
    .dt.days / 30.436875
) < 12
df.loc[mask_under12, 'Percentage utilized in last 12 months'] = 'N/A'

# calculate utilization lag and utilization Status
df['Months elapsed'] = (
    (df['Date Utilization Last Updated'] - df['*Actual Grant Start Date']).dt.days
    / 30.436875
)
df['Total months'] = (
    (df['*Expected Grant Closing Date'] - df['*Actual Grant Start Date']).dt.days
    / 30.436875
)
df['Utilization lag'] = (
    (df['Cumulative Utilized'] / df['Grant Amount'])
    - (df['Months elapsed'] / df['Total months'])
)

def lag_status(lag):
    if lag <= -0.25:
        return 'Delayed'
    elif lag < -0.15:
        return 'Slightly delayed'
    else:
        return 'On track'

df['Utilization Status'] = df['Utilization lag'].apply(lag_status)

# year 1 utilization status
def compute_year1_status(r):
    if pd.isna(r['Grant Approval Date']) or pd.isna(r['Date Utilization Last Updated']):
        return ''
    months = (r['Date Utilization Last Updated'] - r['Grant Approval Date']).days / 30.436875
    if months < 12 or months > 24:
        return ''
    frac = r['Cumulative Utilized'] / r['Grant Amount'] if r['Grant Amount'] else 0
    if frac > 0.1:
        return ''
    if (frac * 12 / months) < 0.1:
        return 'Slow start'
    return ''

df['Year 1 utilization status'] = df.apply(compute_year1_status, axis=1)

# add cumulative percentage column
df['Percentage utilized cumulatively'] = df['Cumulative Utilized'] / df['Grant Amount']

# clean up intermediate cols
df.drop(columns=['Months elapsed', 'Total months'], inplace=True)

# ratings merge
mega = pd.read_excel(mega_file, skiprows=1)
mega.columns = mega.columns.str.strip()
ratings = mega[[
    'Grant ID',
    'Current Secretariat Overall Implementation Progress Rating',
    'Current GA Overall Implementation Progress Rating'
]].rename(columns={
    'Current Secretariat Overall Implementation Progress Rating': 'Current Secretariat implementation rating',
    'Current GA Overall Implementation Progress Rating':        'Current GA implementation rating'
})
df = df.merge(ratings, on='Grant ID', how='left')
df['Current Secretariat implementation rating'] = df['Current Secretariat implementation rating'].fillna('No Rating Available')
df['Current GA implementation rating']         = df['Current GA implementation rating'].fillna('No Rating Available')

# WB exception for progress report status
df['Progress Report Submission Status'] = df.apply(
    lambda r: 'N/A' if r['Grant Agent'] == 'WB' else r['Progress Report Submission Status'],
    axis=1
)

# RM dashboard merge
df_rm = pd.read_excel(rm_dashboard_file)
df_rm.columns = df_rm.columns.str.strip()
df = df.merge(
    df_rm[[
        'Grant ID',
        'Last Finalized Monitoring Update Finalized Date',
        'Link To Last Finalized Monitoring Update'
    ]],
    on='Grant ID', how='left'
).rename(columns={
    'Last Finalized Monitoring Update Finalized Date': 'Last finalized monitoring update date',
    'Link To Last Finalized Monitoring Update':          'Last Finalized Monitoring Update'
})

# blank placeholders for the renamed flag column
df['Delayed in implementation, utilization, report submission and/or start'] = ''
df['CPO flag'] = ''

# populate the renamed “Delayed…” column, now including “Slow start”
df['Delayed in implementation, utilization, report submission and/or start'] = df.apply(
    lambda r: ", ".join(filter(None, [
        "Low utilization" if r['Utilization Status'] == 'Delayed' else "",
        "Overdue report"   if r['Progress Report Submission Status'] == 'Overdue' else "",
        r['Current Secretariat implementation rating']
            if r['Current Secretariat implementation rating'] in ('Moderately Unsatisfactory','Unsatisfactory')
            else "",
        r['Year 1 utilization status']
    ])),
    axis=1
)

# reorder implementation grants columns
impl_order = [
    'Region', 'Country Name', 'Grant ID', 'P#', 'Grant Type', 'Grant Agent',
    'Operating Model', 'Grant Approval Date', '*Actual Grant Start Date',
    '*Actual Grant Closing Date', '*Expected Grant Closing Date', 'Grant Amount',
    'Delayed in implementation, utilization, report submission and/or start',
    'CPO flag',
    'Current GA implementation rating', 'Current Secretariat implementation rating',
    'Last Progress Report Submitted/ Received Date', 'Link to Progress Report',
    'Progress Report Submission Status',
    'Last finalized monitoring update date', 'Last Finalized Monitoring Update',
    'Date Utilization Last Updated', 'Cumulative Utilized',
    'Percentage utilized cumulatively',
    'Percentage utilized in last 12 months', 'Utilization lag',
    'Utilization Status', 'Year 1 utilization status',
    'If Extended How many Times', 'Cumulative extension (in months)'
]
impl_cols = [c for c in impl_order if c in df.columns] + [c for c in df.columns if c not in impl_order]
df_impl = df[impl_cols]


# part 2: all grants sheet
df_all = pd.read_excel(mega_file, skiprows=1)
df_all.columns = df_all.columns.str.strip()
df_all.rename(columns={'Cluster': 'Region'}, inplace=True)
df_all = df_all.loc[:, ~df_all.columns.duplicated(keep='last')]

for c in ["Grant Approval Date", "Actual Grant Start Date", "Expected Grant Closing Date", "Actual Grant Closing Date"]:
    df_all[c] = pd.to_datetime(df_all[c], errors='coerce')

ex_types  = ["A4L","ASA","BELDS","CLADM","CSEF","ESPIG (COVID-19 AF)","GRA","GRESP","KIX","SC"]
ex_status = ["Pipeline","Pipeline Data for Approved Top-up","Proposal Submitted","Cancelled","Financially Closed","Closed"]

mask_all = (
    (df_all["Actual Grant Start Date"] <= as_of_date) &
    (
        (df_all["Actual Grant Closing Date"].isna() & (df_all["Expected Grant Closing Date"] > as_of_date)) |
        (df_all["Actual Grant Closing Date"] > as_of_date)
    ) &
    (~df_all['Grant Type'].isin(ex_types)) &
    (~df_all['Grant Status'].isin(ex_status))
)
cols_all = [
    "Region", "Country Name", "Grant ID", "Grant Status", "Grant Type",
    "Grant Agent", "Grant Operations Officer", "Grant Approval Date",
    "Actual Grant Start Date", "Expected Grant Closing Date", "Actual Grant Closing Date", "Grant Amount"
]
df_all = df_all.loc[mask_all, cols_all]


# part 3: AF sheet (filtered active AF grants)
df_af = pd.read_excel(input_file)
for c in ["*Expected Grant Closing Date", "*Actual Grant Start Date"]:
    df_af[c] = pd.to_datetime(df_af[c], errors='coerce')

mask_af = (
    df_af['Grant Type'].str.contains(r'\(AF\)', na=False) &
    (df_af['*Actual Grant Start Date'] <= as_of_date) &
    (df_af['*Expected Grant Closing Date'] > as_of_date)
)
df_af = df_af.loc[mask_af].reset_index(drop=True)

df_af['Months between start and closing'] = (
    (df_af['*Expected Grant Closing Date'] - df_af['*Actual Grant Start Date']).dt.days
    / 30.436875
)

cols3 = [
    "Cluster", "Country Name", "Grant Type", "Grant ID", "Grant Agent",
    "Grant Approval Date", "*Actual Grant Start Date", "*Expected Grant Closing Date",
    "*Actual Grant Closing Date", "Grant Amount", "If Extended How many Times",
    "Months between start and closing"
]
df_af = df_af[cols3].rename(columns={
    "Cluster": "Region",
    "*Actual Grant Start Date": "Actual Grant Start Date",
    "*Expected Grant Closing Date": "Expected Grant Closing Date",
    "*Actual Grant Closing Date": "Actual Grant Closing Date",
    "If Extended How many Times": "Number of extensions approved"
})
df_af['CPO flag'] = ''
df_af['Other flags'] = ''
df_af['Action'] = ''
order3 = [
    "Region", "Country Name", "Grant Type", "Grant ID", "Grant Agent",
    "Grant Approval Date", "Actual Grant Start Date", "Expected Grant Closing Date",
    "Actual Grant Closing Date", "Grant Amount", "Number of extensions approved",
    "Months between start and closing", "CPO flag", "Other flags", "Action"
]
df_af = df_af[order3]


# part 4: pending sheet
df_p = pd.read_excel(input_file)
df_p['Grant Approval Date'] = pd.to_datetime(df_p['Grant Approval Date'], errors='coerce')
df_p['Months passed since approval'] = (
    (as_of_date - df_p['Grant Approval Date']).dt.days / 30.436875
)
cols4 = [
    "Cluster", "Country Name", "Grant Status", "Grant Type", "P#", "Grant Agent",
    "Grant Operations Officer", "Grant Approval Date", "Original Estimated Grant Start Date",
    "*Estimated Grant Start Date", "Grant Amount", "Months passed since approval", "Notes"
]
df_p = df_p[cols4]
for c in ["Original Estimated Grant Start Date", "*Estimated Grant Start Date"]:
    df_p[c] = pd.to_datetime(df_p[c], errors='coerce')
df_p = df_p[df_p['Grant Status'] == 'Pending']
df_p.rename(columns={'Cluster': 'Region', '*Estimated Grant Start Date': 'Estimated Grant Start Date'}, inplace=True)

# adding months from approval to estimated start date
df_p['Months from approval to est start'] = (
    (df_p['Estimated Grant Start Date'] - df_p['Grant Approval Date']).dt.days
    / 30.436875
).round(1)


order4 = [
    "Region", "Country Name", "Grant Status", "Grant Type", "P#", "Grant Agent",
    "Grant Operations Officer", "Grant Approval Date", "Original Estimated Grant Start Date",
    "Estimated Grant Start Date", "Grant Amount", "Months passed since approval", "Months from approval to est start", "Notes"
]
df_p = df_p[order4]


# save all sheets
output_file = f'Grant monitoring tool as of {as_of_date_input}.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_impl.to_excel(writer, sheet_name='Implementation grants', index=False)
    df_all.to_excel(writer, sheet_name='All grants', index=False)
    df_af.to_excel(writer, sheet_name='AF', index=False)
    df_p.to_excel(writer, sheet_name='Pending', index=False)

print(f"Updated file saved as {output_file}")

/tmp/ipykernel_400/1848548593.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'N/A' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[mask_under12, 'Percentage utilized in last 12 months'] = 'N/A'


Updated file saved as Grant monitoring tool as of 2026-01-31.xlsx


In [ ]:
from google.colab import drive
drive.mount('/content/drive')